In [1]:
import os
import pandas as pd
from glob import glob
from pathlib import Path
from joblib import Parallel, delayed
from multiprocessing import Pool
from functools import partial

import gcsfs
from google.cloud import storage


In [2]:
from pymol import cmd as pmcmd
import tempfile
import gcsfs
from google.cloud import storage
from tmtools import tm_align
from tmtools.io import get_structure, get_residue_data

def get_gcs_path(p):
    p = p.split('gcs/')[1]
    return f"gs://{p}"

def get_rfab_path(abspath, rid):
    return f'{abspath}/extracted_pdbs/{rid}.pdb'

def pmcmd_load_gcs(gcs_uri: str, object_name: str, fs: gcsfs.GCSFileSystem = None):
    """Load a PDB (or any structure file) from GCS into PyMOL."""
    if fs is None:
        fs = gcsfs.GCSFileSystem()
    
    # Preserve the original extension so PyMOL auto-detects format
    ext = os.path.splitext(gcs_uri)[1]  # e.g. '.pdb', '.cif'
    
    with tempfile.NamedTemporaryFile(suffix=ext, delete=False) as tmp:
        fs.get(gcs_uri, tmp.name)
        pmcmd.load(tmp.name, object_name)
        local = tmp.name
    
    os.unlink(local)  # clean up
    return object_name
    
    
def load_pymol_structures(rfab_path, boltz_path, pdb_path):
    "assumes complex folding"
    pmcmd.reinitialize()
    pmcmd.load(rfab_path, 'rfab')
    pmcmd_load_gcs(boltz_path, 'boltz')
    pmcmd.load(pdb_path, 'reference') # reference here is only used for comparing folds of target

In [3]:
# PyMol helpers
from scipy.spatial.distance import cdist
from typing import List, Any, Dict, Optional, Tuple
import numpy as np
from pymol import cmd as pmcmd
from frozendict import frozendict
from Bio.PDB.Polypeptide import aa3, aa1
AA_3TO1_MAP=frozendict({k:v for k,v in zip(aa3, aa1)})

def extract_interface_residues(selection: str) -> List[str]:
    """Extract residue identifiers in single-letter format."""
    residues = []
    pmcmd.iterate(
        f'{selection} and name CA',
        'residues.append(f"{m.get(resn, \'X\')}{resi}")',
        space={'residues': residues, 'm': AA_3TO1_MAP}
    )
    return residues


def extract_residues_with_all_coords(selection: str) -> Tuple[List[str], List[np.ndarray]]:
    """
    Extract residue identifiers and ALL heavy atom coordinates per residue.
    E.g. not only computes Ca distances
    """
    residue_data = {}

    def collect(resn, resi, x, y, z, m=AA_3TO1_MAP):
        key = f"{m.get(resn, 'X')}{resi}"
        if key not in residue_data:
            residue_data[key] = []
        residue_data[key].append([x, y, z])

    pmcmd.iterate_state(
        1,
        f'{selection} and not elem H',  # Heavy atoms only
        'collect(resn, resi, x, y, z)',
        space={'collect': collect, 'AA_3TO1_MAP': AA_3TO1_MAP}
    )

    residues = list(residue_data.keys())
    coords = [np.array(residue_data[r]) for r in residues]
    return residues, coords


def compute_min_distances_all_atoms(
        coords_a: List[np.ndarray],
        coords_b: List[np.ndarray]
) -> Tuple[np.ndarray, np.ndarray]:
    """Compute minimum distance between any atom pair for each residue pair."""
    from scipy.spatial.distance import cdist

    n_a, n_b = len(coords_a), len(coords_b)

    # ── Guard against empty inputs ──────────────────────────────────
    if n_a == 0 or n_b == 0:
        return np.array([], dtype=float), np.array([], dtype=int)
    # ────────────────────────────────────────────────────────────────

    min_dist_matrix = np.zeros((n_a, n_b))

    for i, ca in enumerate(coords_a):
        for j, cb in enumerate(coords_b):
            min_dist_matrix[i, j] = cdist(ca, cb).min()

    return min_dist_matrix.min(axis=1), min_dist_matrix.argmin(axis=1)


def extract_residues_with_coords(selection: str) -> Tuple[List[str], np.ndarray]:
    """Extract residue identifiers and their CA coordinates."""
    residues = []
    coords = []
    pmcmd.iterate_state(
        1,  # State 1 (first/only state)
        f'{selection} and name CA',
        'residues.append(f"{m.get(resn, \'X\')}{resi}"); coords.append((x, y, z))',
        space={'residues': residues, 'coords': coords, 'm': AA_3TO1_MAP}
    )
    return residues, np.array(coords)


def compute_min_distances_scipy(
            coords_a: np.ndarray,
            coords_b: np.ndarray
    ) -> Tuple[np.ndarray, np.ndarray]:
    """
    Compute minimum distance from each residue in A to any residue in B using scipy.cdist

    Returns:
        min_distances: Array of minimum distances for each residue in A
        closest_indices: Index in B of the closest residue for each in A
    """
    dist_matrix = cdist(coords_a, coords_b, metric='euclidean')
    return dist_matrix.min(axis=1), dist_matrix.argmin(axis=1)


def get_complex_interface(
        object_name: str,
        chain_h: str = 'H',
        chain_tg: str = 'T',
        threshold: float = 5.0,
        use_all_atoms: bool = True
) -> Dict:
    """
    Extract interface with accurate distance measurement.
    assumes pdb is already loaded and have names: 
    - "boltz" (chain A, chain B)
    - "rfab" (chain T, chain H)
    """
    if object_name == 'rfab':
        chain_h, chain_tg = 'H', 'T'
    elif object_name == 'boltz':
        chain_h, chain_tg = 'B', 'A'
        
    ab_string = f'chain {chain_h}'
    pmcmd.select('epitope', f'byres (chain {chain_tg} within {threshold} of ({ab_string}))')
    pmcmd.select('paratope', f'byres (({ab_string}) within {threshold} of chain {chain_tg})')

    # ── Guard: no interface found ───────────────────────────────────
    epitope_count = pmcmd.count_atoms('epitope and name CA')
    paratope_count = pmcmd.count_atoms('paratope and name CA')

    if epitope_count == 0 or paratope_count == 0:
        return {
            'epitope_residues': [],
            'epitope_distances': [],
            'epitope_closest_partner': [],
            'paratope_residues': [],
            'paratope_distances': [],
            'paratope_closest_partner': [],
        }
    # ────────────────────────────────────────────────────────────────

    if use_all_atoms:
        epitope_residues, epitope_coords = extract_residues_with_all_coords('epitope')
        paratope_residues, paratope_coords = extract_residues_with_all_coords('paratope')
        epitope_min_dist, epitope_closest_idx = compute_min_distances_all_atoms(
            epitope_coords, paratope_coords
        )
        paratope_min_dist, paratope_closest_idx = compute_min_distances_all_atoms(
            paratope_coords, epitope_coords
        )
    else:
        epitope_residues, epitope_coords = extract_residues_with_coords('epitope')
        paratope_residues, paratope_coords = extract_residues_with_coords('paratope')
        epitope_min_dist, epitope_closest_idx = compute_min_distances_scipy(
            epitope_coords, paratope_coords
        )
        paratope_min_dist, paratope_closest_idx = compute_min_distances_scipy(
            paratope_coords, epitope_coords
        )

    return {
        'epitope_residues': epitope_residues,
        'epitope_distances': epitope_min_dist.tolist(),
        'epitope_closest_partner': [paratope_residues[i] for i in epitope_closest_idx],
        'paratope_residues': paratope_residues,
        'paratope_distances': paratope_min_dist.tolist(),
        'paratope_closest_partner': [epitope_residues[i] for i in paratope_closest_idx],
    }

In [24]:
a,b = df1x[['vh', 'mab_id']].values

ValueError: too many values to unpack (expected 2)

In [7]:
PATH = '/home/JV11_DK2/ab-develop/projects/uc_denovo_vhh/RFantibody/'
df1x = pd.read_csv(f'{PATH}/data/05_results/rfab_hyp03_1XIW_filtered.csv')
df1s_cor = pd.read_csv(f'{PATH}/data/05_results/rfab_hyp03_1SY6_filtered_curated_epitope.csv')
df1s_wro = pd.read_csv(f'{PATH}/data/05_results/rfab_hyp03_1SY6_filtered_wrong_epitope.csv')
samples_1x = df1x.query('interaction_pae < 5.0 and target_aligned_cdr_rmsd < 2.0 and framework.str.contains("3eak")')
sample = samples_1x.iloc[0]

rfab_path = get_rfab_path(sample['fullpath'], sample['rfab_id'])
boltz_path = get_gcs_path(sample['complex_pdb_path'])
pdb_path = f'{PATH}/data/02_intermediate/target/CD3e_1XIW_processed.pdb'
load_pymol_structures(rfab_path, boltz_path, pdb_path)

In [14]:
list(df1x.columns)[:30]

['Unnamed: 0',
 'timestamp',
 'framework',
 'target',
 'hotspots',
 'rfab_id',
 'vh',
 't',
 'H1_start',
 'H1_end',
 'H2_start',
 'H2_end',
 'H3_start',
 'H3_end',
 'interaction_pae',
 'pae',
 'pred_lddt',
 'target_aligned_antibody_rmsd',
 'target_aligned_cdr_rmsd',
 'framework_aligned_antibody_rmsd',
 'framework_aligned_cdr_rmsd',
 'framework_aligned_H1_rmsd',
 'framework_aligned_H2_rmsd',
 'framework_aligned_H3_rmsd',
 'framework_aligned_L1_rmsd',
 'framework_aligned_L2_rmsd',
 'framework_aligned_L3_rmsd',
 'epitope',
 'mab_id',
 'vh_hash']

In [21]:
df1x.query('interaction_pae <= 5.0 and pred_lddt >= 0.9 and framework_aligned_antibody_rmsd <= 1.0').groupby('framework').count()

,Unnamed: 0,timestamp,target,hotspots,rfab_id,vh,t,H1_start,H1_end,H2_start,...,mean_interface_resnum,ml_dG_kcal_mol,ml_Kd_nM,fullpath,epitope_residues,epitope_distances,epitope_closest_partner,paratope_residues,paratope_distances,paratope_closest_partner
framework,,,,,,,,,,,,,,,,,,,,,
3eak_hlt,81,81,81,81,81,81,81,81,81,81,...,81,81,81,81,81,81,81,81,81,81
7dv4_hlt,63,63,63,63,63,63,63,63,63,63,...,63,56,56,63,63,63,63,63,63,63
7eow_hlt,191,191,191,191,191,191,191,191,191,191,...,191,155,155,191,191,191,191,191,191,191
7xl0_hlt,86,86,86,86,86,86,86,86,86,86,...,86,86,86,86,86,86,86,86,86,86
8coh_hlt,123,123,123,123,123,123,123,123,123,123,...,123,123,123,123,123,123,123,123,123,123
8q6r_hlt,214,214,214,214,214,214,214,214,214,214,...,214,214,214,214,214,214,214,214,214,214
8z8m_hlt,74,74,74,74,74,74,74,74,74,74,...,74,74,74,74,74,74,74,74,74,74
8z8v_hlt,12,12,12,12,12,12,12,12,12,12,...,12,12,12,12,12,12,12,12,12,12


In [9]:
from tmtools import tm_align, TMResult

# get epitopes and do alignment RMSD

## RMSD

In [22]:
from Bio.PDB import PDBParser
from collections  import defaultdict
from Bio.SeqUtils import seq1

def parse_sequences_from_pdb(pdb_path: Path | str, fs: gcsfs.GCSFileSystem = None) -> Dict[str, str]:
    """
    Extract per-chain sequences from a PDB using Bio.PDB.
    Accepts either a local path or a gs:// URI.
    Uses first model only; skips hetero residues.
    """
    is_gcs = str(pdb_path).startswith("gs://")

    if is_gcs:
        gcs_uri = str(pdb_path)
        if fs is None:
            fs = gcsfs.GCSFileSystem()
        ext = os.path.splitext(gcs_uri)[1]
        stem = os.path.splitext(os.path.basename(gcs_uri))[0]
        tmp = tempfile.NamedTemporaryFile(suffix=ext, delete=False)
        tmp.close()
        fs.get(gcs_uri, tmp.name)
        local_path = Path(tmp.name)
    else:
        local_path = Path(pdb_path)
        stem = local_path.stem

    try:
        parser = PDBParser(QUIET=True)
        with local_path.open("r", encoding="latin-1", errors="replace", newline="") as handle:
            structure = parser.get_structure(stem, handle)

        chain_res: Dict[str, List[Tuple[int, str, str]]] = defaultdict(list)
        model = next(structure.get_models())

        for chain in model:
            for res in chain:
                hetflag, resseq, icode = res.id
                if hetflag != " ":
                    continue
                try:
                    aa = seq1(res.get_resname(), custom_map={"MSE": "M"})
                except Exception:
                    aa = "X"
                chain_res[chain.id].append((resseq, str(icode).strip(), aa))

        seqs: Dict[str, str] = {}
        for ch, items in chain_res.items():
            items.sort(key=lambda x: (x[0], x[1]))
            seqs[ch] = "".join(aa for _, _, aa in items)
        return seqs
    finally:
        if is_gcs:
            os.unlink(local_path)
            
def get_pymol_rmsd():
    res_h = pmcmd.align('boltz and chain B and name CA','rfab and chain H and name CA', )
    res_t = pmcmd.align('boltz and chain A and name CA','rfab and chain T and name CA', )
    res_c = pmcmd.align('boltz and name CA', 'rfab and name CA')
    # aligned Ab
    res_h = {f'h_{k}':v for k,v in zip(['rmsd_ref', 'n_aligned_atoms_ref', 'n_cycle', 'rmsd_raw', 'n_aligned_atoms_raw','aligned_score','n_aligned_residues'], 
                                res_h) if k not in ['n_cycle', 'aligned_score']}
    # aligned Ab
    res_t = {f't_{k}':v for k,v in zip(['rmsd_ref', 'n_aligned_atoms_ref', 'n_cycle', 'rmsd_raw', 'n_aligned_atoms_raw','aligned_score','n_aligned_residues'], 
                                res_t) if k not in ['n_cycle', 'aligned_score']}
    
    res_c = {f'c_{k}':v for k,v in zip(['rmsd_ref', 'n_aligned_atoms_ref', 'n_cycle', 'rmsd_raw', 'n_aligned_atoms_raw','aligned_score','n_aligned_residues'], 
                                res_c) if k not in ['n_cycle', 'aligned_score']}
    res_c.update(res_h)
    res_c.update(res_t)
    # return res_h, res_t, res_complex
    return res_c

def get_pymol_rmsd_reference(select_name, select_chains, ref_name='reference', ref_chains='A,B'):
    # Only does the target here
    ref_str = 'chain ' +' or chain '.join(ref_chains.split(','))
    select_str = 'chain ' +' or chain '.join(select_chains.split(','))

    res = pmcmd.align(f'{ref_name} and ({ref_str}) and name CA', f'{select_name} and ({select_str}) and name CA')
    return {f'h_{k}':v for k,v in zip(['rmsd_ref', 'n_aligned_atoms_ref', 'n_cycle', 'rmsd_raw', 'n_aligned_atoms_raw','aligned_score','n_aligned_residues'], 
                            res) if k not in ['n_cycle', 'aligned_score']}

In [42]:
get_pymol_rmsd_reference('boltz', 'A')

{'h_rmsd_ref': 0.41512569785118103,
 'h_n_aligned_atoms_ref': 128,
 'h_rmsd_raw': 0.9902764558792114,
 'h_n_aligned_atoms_raw': 157,
 'h_n_aligned_residues': 157}

In [43]:
get_pymol_rmsd_reference('rfab', 'T')

{'h_rmsd_ref': 0.1441897302865982,
 'h_n_aligned_atoms_ref': 150,
 'h_rmsd_raw': 1.0050371885299683,
 'h_n_aligned_atoms_raw': 157,
 'h_n_aligned_residues': 157}

## TM score

In [70]:
from tmtools import tm_align
from tmtools.io import get_structure, get_residue_data

def load_tm_structure(rfab_path, boltz_path, pdb_path):
    rfab_struct = get_structure(rfab_path)
    pdb_struct = get_structure(pdb_path)
    # Load Boltzgen pdb from bucket
    fs = gcsfs.GCSFileSystem()
    # Preserve the original extension so PyMOL auto-detects format
    ext = os.path.splitext(boltz_path)[1]  # e.g. '.pdb', '.cif'
    with tempfile.NamedTemporaryFile(suffix=ext, delete=False) as tmp:
        fs.get(boltz_path, tmp.name)
        boltz_struct = get_structure(tmp.name)
        local = tmp.name

    return rfab_struct, boltz_struct, pdb_struct

def align_tm_scores(rfab_struct, boltz_struct):
    """
    Rfab:  Target (T), Heavy (H)
    Boltz: Target (A), Heavy (B)

    Returns per-chain TM-scores + RMSD, and a 'Complex' entry computed by
    concatenating the chains in a matched order on both sides.
    """
    rfab_chains = {x.id: x for x in rfab_struct.get_chains()}
    boltz_chains = {x.id: x for x in boltz_struct.get_chains()}

    # Keep an ordered mapping so concatenation is deterministic and consistent
    chain_ids = [
        ('Target', 'T', 'A'),
        ('VHH',    'H', 'B'),
    ]

    res = {}
    rfab_coords_all, rfab_seq_all = [], []
    boltz_coords_all, boltz_seq_all = [], []

    for label, rfab_id, boltz_id in chain_ids:
        c1, s1 = get_residue_data(rfab_chains[rfab_id])
        c2, s2 = get_residue_data(boltz_chains[boltz_id])

        score = tm_align(c1, c2, s1, s2)
        res[f'{label}_tm_norm_c1'] = score.tm_norm_chain1
        res[f'{label}_tm_norm_c2'] = score.tm_norm_chain2
        res[f'{label}_rmsd']       = score.rmsd

        rfab_coords_all.append(c1);  rfab_seq_all.append(s1)
        boltz_coords_all.append(c2); boltz_seq_all.append(s2)

    # --- Full complex: concatenate in the same chain order on both sides ---
    c1_full = np.concatenate(rfab_coords_all, axis=0)
    c2_full = np.concatenate(boltz_coords_all, axis=0)
    s1_full = "".join(rfab_seq_all)
    s2_full = "".join(boltz_seq_all)

    complex_score = tm_align(c1_full, c2_full, s1_full, s2_full)
    res['Complex_tm_norm_c1'] = complex_score.tm_norm_chain1
    res['Complex_tm_norm_c2'] = complex_score.tm_norm_chain2
    res['Complex_rmsd']       = complex_score.rmsd

    return res

def align_tm_score_ref(select_struct, select_chains, ref_struct):
    """
    Rfab:  Target (T), Heavy (H)
    Boltz: Target (A), Heavy (B)

    Returns per-chain TM-scores + RMSD, and a 'Complex' entry computed by
    concatenating the chains in a matched order on both sides.
    """
    # c1, s1 C1 == ref
    c_ref, s_ref = get_residue_data(ref_struct)
    tmp_dict = {x.id: x for x in select_struct.get_chains()}
    if len(select_chains.split(','))==1:
        # c2, s2
        c_sel, s_sel = get_residue_data(tmp_dict[select_chains])
    else:
        c_sel, s_sel = [], []
        for ch in select_chains.split(','):
            c_tmp, s_tmp = get_residue_data(tmp_dict[ch])
            c_sel.append(c_tmp)
            s_sel.append(s_tmp)
        c_sel = np.concatenate(c_sel, axis=0)
        s_sel = "".join(s_sel)
        
    # c1, c2, s1, s2
    score = tm_align(c_ref, c_sel, s_ref, s_sel)
    res = {'tm_score_c1': score.tm_norm_chain1,
           'tm_score_': score.tm_norm_chain2,
           'rmsd':  score.rmsd}
    return res

In [71]:
rfab_struct, boltz_struct, pdb_struct = load_tm_structure(rfab_path, boltz_path, pdb_path)
align_tm_score_ref(rfab_struct, 'T', pdb_struct), align_tm_score_ref(boltz_struct, 'A', pdb_struct)

({'tm_score_c1': 0.9921267081925879,
  'tm_score_': 0.9921267081925879,
  'rmsd': 0.18254023457427926},
 {'tm_score_c1': 0.9671745259849491,
  'tm_score_': 0.7908120795464239,
  'rmsd': 0.9902764503204073})

In [72]:
px_dimer_msa_struct = get_structure('../data/sandbox/Protenix_rfab_dimer_seed10_sample0_msa.cif', format='mmcif')
px_dimer_nomsa_struct = get_structure('../data/sandbox/Protenix_rfab_dimer_seed10_sample0_NOMSA.cif', format='mmcif')
px_linker_msa_struct = get_structure('../data/sandbox/Protenix_rfab_linker_seed10_sample0_msa.cif', format='mmcif')

In [75]:
print(align_tm_score_ref(rfab_struct, 'T', pdb_struct))
print(align_tm_score_ref(boltz_struct, 'A', pdb_struct))
print(align_tm_score_ref(px_dimer_msa_struct, 'T1,T2', pdb_struct))
print(align_tm_score_ref(px_dimer_nomsa_struct, 'T1,T2' , pdb_struct))
print(align_tm_score_ref(px_linker_msa_struct, 'T' , pdb_struct), )

{'tm_score_c1': 0.9921267081925879, 'tm_score_': 0.9921267081925879, 'rmsd': 0.18254023457427926}
{'tm_score_c1': 0.9671745259849491, 'tm_score_': 0.7908120795464239, 'rmsd': 0.9902764503204073}
{'tm_score_c1': 0.9735427064396288, 'tm_score_': 0.9735427064396288, 'rmsd': 0.9741491739786557}
{'tm_score_c1': 0.29158456426794116, 'tm_score_': 0.29158456426794116, 'rmsd': 4.401418191434938}
{'tm_score_c1': 0.9629232973314742, 'tm_score_': 0.7874028056225195, 'rmsd': 1.157120255138693}


In [55]:
align_tm_score_ref(rfab_struct, 'T', pdb_struct), align_tm_score_ref(boltz_struct, 'A', pdb_struct)

[]

In [239]:
# Take one epitope --> find the same numbering in the other struct
# --> compute sasa for this same epitope
# --> they should have similar coverage if both epitopes are same
# if not, one will have more accessible residues
sample['epitope_residues'] 

"['E150', 'L152', 'W153', 'Q154', 'D157', 'K158', 'N159', 'I160', 'G161', 'G162', 'D163', 'E164', 'S171', 'Y193', 'P194', 'R195', 'G196', 'S197', 'K198', 'P199', 'E200']"

In [237]:
get_rfab_path(sample['fullpath'], sample['rfab_id']), get_gcs_path(sample['complex_pdb_path'])

('/home/JV11_DK2/ab-develop/projects/uc_denovo_vhh/RFantibody/data/04_rfab/2604XX_CD3e_1XIW_1XIW/1XIW/260414_112215_runqv_fw3eak_HLT_tgCD3e_1XIW_processed_hsA35A44A45A47A48/extracted_pdbs/samples_design_12_dldesign_11_best.pdb',
 'gs://em52-ab-develop-analytics-prod-f684/data/denovo-design/snakemake/data/02_intermediate/evaluation-engine-outputs/260419_CD3e_1XIW_hypothesis03_rfab/36000-37000/36000_37000/complex_structures/boltz_results_complex/predictions/binder_00513_881da32c/binder_00513_881da32c_model_0.pdb')

In [272]:
raw_path = '../data/02_intermediate/target/CD3e_1XIW_processed.pdb'
rfab_path = get_rfab_path(sample['fullpath'], sample['rfab_id'])
boltz_path = get_gcs_path(sample['complex_pdb_path'])
load_pymol_structures(rfab_path, boltz_path)
# result_heavy, result_target, result_complex = get_pymol_rmsd()
result_rmsd = get_pymol_rmsd()

rfab_struct, boltz_struct = load_tm_structure(rfab_path, boltz_path)
results_tm = align_tm_scores(rfab_struct, boltz_struct)

In [276]:
def fullpipe(row):
    rpath = get_rfab_path(row['fullpath'], row['rfab_id'])
    bpath = get_gcs_path(row['complex_pdb_path'])
    load_pymol_structures(rpath, bpath)
    results = get_pymol_rmsd()
    rstruc, bstruc = load_tm_structure(rpath, bpath)
    results.update(align_tm_scores(rstruc, bstruc))
    return results
fullpipe(sample)
    

{'c_rmsd_ref': 0.4721844792366028,
 'c_n_aligned_atoms_ref': 119,
 'c_rmsd_raw': 1.6909040212631226,
 'c_n_aligned_atoms_raw': 157,
 'c_n_aligned_residues': 157,
 'h_rmsd_ref': 0.16553592681884766,
 'h_n_aligned_atoms_ref': 101,
 'h_rmsd_raw': 0.334231972694397,
 'h_n_aligned_atoms_raw': 126,
 'h_n_aligned_residues': 126,
 't_rmsd_ref': 0.47218456864356995,
 't_n_aligned_atoms_ref': 119,
 't_rmsd_raw': 1.6909040212631226,
 't_n_aligned_atoms_raw': 157,
 't_n_aligned_residues': 157,
 'Target_tm_norm_c1': 0.9510522493207348,
 'Target_tm_norm_c2': 0.7790338355600108,
 'Target_rmsd': 1.1056040720152385,
 'VHH_tm_norm_c1': 0.9937221364934502,
 'VHH_tm_norm_c2': 0.9937221364934502,
 'VHH_rmsd': 0.33423198011501404,
 'Complex_tm_norm_c1': 0.5365117648028862,
 'Complex_tm_norm_c2': 0.4771352808515154,
 'Complex_rmsd': 1.1056040720153228}

In [ ]:
from tqdm.auto import tqdm
out = Parallel(n_jobs=16)(delayed(fullpipe)(row) for _, row in tqdm(df1x.iterrows()))

/opt/conda/envs/ada/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2272it [01:34, 24.39it/s]

## Aligning between different PDBs to get epitopes


In [ ]:
# 1SY6 : Epsilon is on the second part and starts at index 82
# 1XIW : Epsilon is on the first path and ends at 95. We also added a linker on 1XIW that goes from 95 to 127

In [252]:
import os
import tempfile
from contextlib import contextmanager
from dataclasses import dataclass
from pathlib import Path
from collections import defaultdict
from typing import Dict, List, Tuple, Optional, Iterator, Union

import gcsfs
from Bio.PDB import PDBParser
from Bio.SeqUtils import seq1
from Bio import Align


ResKey = Tuple[int, str]  # (author_resnum, icode)
PathLike = Union[str, Path]


# ---------------------------------------------------------------------------
# Source abstraction: local path OR gs:// URI, plus which chain to use
# ---------------------------------------------------------------------------

@dataclass
class StructureSource:
    """A PDB/CIF source plus the chain of interest within it."""
    path: str                 # local path or 'gs://...' URI
    chain_id: str             # chain ID of the target within THIS file
    label: Optional[str] = None  # optional human-readable tag for logs

    @property
    def is_gcs(self) -> bool:
        return str(self.path).startswith("gs://")


@contextmanager
def _local_copy(path: PathLike, fs: Optional[gcsfs.GCSFileSystem] = None) -> Iterator[Path]:
    """
    Yield a local Path for `path`. If `path` is a gs:// URI, download to a
    temp file (preserving extension) and clean up afterwards.
    """
    s = str(path)
    if s.startswith("gs://"):
        if fs is None:
            fs = gcsfs.GCSFileSystem()
        ext = os.path.splitext(s)[1]
        tmp = tempfile.NamedTemporaryFile(suffix=ext, delete=False)
        tmp.close()
        try:
            fs.get(s, tmp.name)
            yield Path(tmp.name)
        finally:
            try:
                os.unlink(tmp.name)
            except FileNotFoundError:
                pass
    else:
        yield Path(s)


# ---------------------------------------------------------------------------
# Sequence extraction
# ---------------------------------------------------------------------------

def extract_chain_sequence_with_map(
    source: StructureSource,
    fs: Optional[gcsfs.GCSFileSystem] = None,
) -> Tuple[str, List[ResKey]]:
    """
    Return (sequence, residue_keys) for source.chain_id.
    residue_keys[i] is (author_resnum, icode) for the i-th residue in the
    returned sequence.
    """
    with _local_copy(source.path, fs=fs) as local_path:
        parser = PDBParser(QUIET=True)
        with local_path.open("r", encoding="latin-1", errors="replace", newline="") as fh:
            structure = parser.get_structure(local_path.stem, fh)

        model = next(structure.get_models())
        if source.chain_id not in [c.id for c in model]:
            available = [c.id for c in model]
            raise KeyError(
                f"Chain {source.chain_id!r} not found in "
                f"{source.label or source.path}. Available: {available}"
            )
        chain = model[source.chain_id]

        residues = []
        for res in chain:
            hetflag, resseq, icode = res.id
            if hetflag != " ":
                continue
            try:
                aa = seq1(res.get_resname(), custom_map={"MSE": "M"})
            except Exception:
                aa = "X"
            residues.append((resseq, str(icode).strip(), aa))

    residues.sort(key=lambda x: (x[0], x[1]))
    seq = "".join(aa for _, _, aa in residues)
    keys: List[ResKey] = [(r, ic) for r, ic, _ in residues]
    return seq, keys


def parse_all_chain_sequences(
    path: PathLike,
    fs: Optional[gcsfs.GCSFileSystem] = None,
) -> Dict[str, str]:
    """
    Convenience: extract sequences for ALL chains (first model).
    Useful for discovering chain IDs when you don't know them.
    """
    with _local_copy(path, fs=fs) as local_path:
        parser = PDBParser(QUIET=True)
        with local_path.open("r", encoding="latin-1", errors="replace", newline="") as fh:
            structure = parser.get_structure(local_path.stem, fh)

        chain_res: Dict[str, List[Tuple[int, str, str]]] = defaultdict(list)
        model = next(structure.get_models())
        for chain in model:
            for res in chain:
                hetflag, resseq, icode = res.id
                if hetflag != " ":
                    continue
                try:
                    aa = seq1(res.get_resname(), custom_map={"MSE": "M"})
                except Exception:
                    aa = "X"
                chain_res[chain.id].append((resseq, str(icode).strip(), aa))

    seqs: Dict[str, str] = {}
    for ch, items in chain_res.items():
        items.sort(key=lambda x: (x[0], x[1]))
        seqs[ch] = "".join(aa for _, _, aa in items)
    return seqs


# ---------------------------------------------------------------------------
# Alignment-based residue mapping
# ---------------------------------------------------------------------------

def _make_aligner() -> Align.PairwiseAligner:
    aligner = Align.PairwiseAligner()
    aligner.mode = "global"
    aligner.match_score = 2
    aligner.mismatch_score = -1
    aligner.open_gap_score = -5
    aligner.extend_gap_score = -0.5
    # Free end gaps => trimmed termini / extra linker regions don't hurt
    aligner.target_end_gap_score = 0.0
    aligner.query_end_gap_score = 0.0
    return aligner


def align_positions(seq_a: str, seq_b: str) -> Dict[int, int]:
    """0-based index-in-A -> 0-based index-in-B, for matched columns only."""
    aligner = _make_aligner()
    aln = aligner.align(seq_a, seq_b)[0]
    a_blocks, b_blocks = aln.aligned
    mapping: Dict[int, int] = {}
    for (a_start, a_end), (b_start, b_end) in zip(a_blocks, b_blocks):
        for off in range(a_end - a_start):
            mapping[a_start + off] = b_start + off
    return mapping

from typing import Iterable, Union

# Accept any of: (resnum,), (resnum, icode), ('E', 150), ('E', 150, ''), or just 150
RawResSpec = Union[int, Tuple]

def _normalize_residue_spec(spec: RawResSpec) -> Tuple[Optional[str], int, str]:
    """
    Returns (aa_letter_or_None, resnum:int, icode:str).
    Accepts:
        150
        (150,)
        (150, 'A')          # resnum, icode
        ('E', 150)          # aa, resnum
        ('E', 150, 'A')     # aa, resnum, icode
    """
    if isinstance(spec, int):
        return None, spec, ""
    if isinstance(spec, tuple):
        if len(spec) == 1:
            return None, int(spec[0]), ""
        if len(spec) == 2:
            a, b = spec
            if isinstance(a, str) and a.isalpha() and len(a) == 1:
                return a.upper(), int(b), ""
            # otherwise assume (resnum, icode)
            return None, int(a), str(b)
        if len(spec) == 3:
            aa, num, ic = spec
            return (str(aa).upper() if aa else None), int(num), str(ic or "")
    raise ValueError(f"Unrecognized residue spec: {spec!r}")


def map_residues_between_sources(
    src_a: StructureSource,
    src_b: StructureSource,
    residues_in_a: Iterable[RawResSpec],
    fs: Optional[gcsfs.GCSFileSystem] = None,
    strict_identity_check: bool = True,
) -> Dict[Tuple[Optional[str], int, str], Optional[Tuple[str, int, str]]]:
    """
    Map residues from src_a's chain to src_b's chain via pairwise sequence
    alignment. Returns a dict keyed by the *normalized* input spec
    (aa_or_None, resnum, icode) -> (aa_in_B, resnum_in_B, icode_in_B) or None.
    """
    seq_a, keys_a = extract_chain_sequence_with_map(src_a, fs=fs)   # keys_a: List[(int, str)]
    seq_b, keys_b = extract_chain_sequence_with_map(src_b, fs=fs)

    # Index A by (resnum, icode) AND by resnum-only (first occurrence) for convenience
    idx_a_full: Dict[Tuple[int, str], int] = {k: i for i, k in enumerate(keys_a)}
    idx_a_num: Dict[int, int] = {}
    for i, (num, ic) in enumerate(keys_a):
        idx_a_num.setdefault(num, i)

    idx_map = align_positions(seq_a, seq_b)

    out = {}
    for spec in residues_in_a:
        aa, num, ic = _normalize_residue_spec(spec)
        key_norm = (aa, num, ic)

        i = idx_a_full.get((num, ic))
        if i is None and ic == "":
            i = idx_a_num.get(num)

        if i is None:
            print(f"[warn] residue {spec!r} not found in {src_a.label or src_a.path} "
                  f"(chain {src_a.chain_id}). "
                  f"Chain spans {keys_a[0][0]}..{keys_a[-1][0]}.")
            out[key_norm] = None
            continue

        if strict_identity_check and aa is not None and seq_a[i] != aa:
            print(f"[warn] residue {spec!r}: expected {aa} at A-position "
                  f"{keys_a[i]}, found {seq_a[i]}. Wrong chain?")
            # still proceed with the mapping

        j = idx_map.get(i)
        if j is None:
            out[key_norm] = None
        else:
            num_b, ic_b = keys_b[j]
            out[key_norm] = (seq_b[j], num_b, ic_b)
    return out

In [253]:

rfab = StructureSource(
    path=rfab_path,
    chain_id="T",
    label="RFantibody",
)

boltz = StructureSource(
    path=boltz_path,
    chain_id="A",
    label="Boltz",
)

ref = StructureSource(
    path="/home/JV11_DK2/ab-develop/projects/uc_denovo_vhh/RFantibody/data/02_intermediate/target/CD3e_1XIW_processed.pdb",
    chain_id="A",            # whatever the native target chain is
    label="1XIW",
)

epitopes = [(x[0], int(x[1:])) for x in eval(sample['epitope_residues'])]
a_to_b = map_residues_between_sources(rfab, boltz, epitopes, fs=None)
a_to_c = map_residues_between_sources(rfab, ref,   epitopes, fs=None)

for k, v in a_to_b.items():
    print(f"RFab {k} -> Boltz {v}")
for k, v in a_to_c.items():
    print(f"RFab {k} -> Ref   {v}")

RFab ('E', 150, '') -> Boltz ('E', 24, '')
RFab ('L', 152, '') -> Boltz ('L', 26, '')
RFab ('W', 153, '') -> Boltz ('W', 27, '')
RFab ('Q', 154, '') -> Boltz ('Q', 28, '')
RFab ('D', 157, '') -> Boltz ('D', 31, '')
RFab ('K', 158, '') -> Boltz ('K', 32, '')
RFab ('N', 159, '') -> Boltz ('N', 33, '')
RFab ('I', 160, '') -> Boltz ('I', 34, '')
RFab ('G', 161, '') -> Boltz ('G', 35, '')
RFab ('G', 162, '') -> Boltz ('G', 36, '')
RFab ('D', 163, '') -> Boltz ('D', 37, '')
RFab ('E', 164, '') -> Boltz ('E', 38, '')
RFab ('S', 171, '') -> Boltz ('S', 45, '')
RFab ('Y', 193, '') -> Boltz ('Y', 67, '')
RFab ('P', 194, '') -> Boltz ('P', 68, '')
RFab ('R', 195, '') -> Boltz ('R', 69, '')
RFab ('G', 196, '') -> Boltz ('G', 70, '')
RFab ('S', 197, '') -> Boltz ('S', 71, '')
RFab ('K', 198, '') -> Boltz ('K', 72, '')
RFab ('P', 199, '') -> Boltz ('P', 73, '')
RFab ('E', 200, '') -> Boltz ('E', 74, '')
RFab ('E', 150, '') -> Ref   ('E', 35, '')
RFab ('L', 152, '') -> Ref   ('L', 37, '')
RFab ('W', 

/opt/conda/envs/ada/lib/python3.13/site-packages/Bio/Align/__init__.py:4414: BiopythonDeprecationWarning: The attribute 'target_end_gap_score' was renamed to 'end_insertion_score'. This was done to be consistent with the
AlignmentCounts object returned by the .counts method of an Alignment object.
  warnings.warn(
/opt/conda/envs/ada/lib/python3.13/site-packages/Bio/Align/__init__.py:4414: BiopythonDeprecationWarning: The attribute 'query_end_gap_score' was renamed to 'end_deletion_score'. This was done to be consistent with the
AlignmentCounts object returned by the .counts method of an Alignment object.
  warnings.warn(
/opt/conda/envs/ada/lib/python3.13/site-packages/Bio/Align/__init__.py:4414: BiopythonDeprecationWarning: The attribute 'target_end_gap_score' was renamed to 'end_insertion_score'. This was done to be consistent with the
AlignmentCounts object returned by the .counts method of an Alignment object.
  warnings.warn(
/opt/conda/envs/ada/lib/python3.13/site-packages/Bio/A

In [256]:
sample['rfab_id']

'samples_design_12_dldesign_11_best'

In [254]:
get_complex_interface('boltz', 'B', 'A')

{'epitope_residues': ['E57',
  'L58',
  'R83',
  'R85',
  'E134',
  'L135',
  'E136',
  'D137',
  'R169',
  'I170',
  'S193'],
 'epitope_distances': [3.758690102678135,
  3.7328182762800606,
  2.7155682859716204,
  3.111731000790443,
  4.821996595294532,
  4.397913848852113,
  3.259092485663634,
  3.3538943482583052,
  2.857378842525704,
  4.487088198841265,
  3.7035134681724515],
 'epitope_closest_partner': ['D107',
  'P104',
  'S106',
  'L110',
  'S106',
  'D63',
  'A62',
  'L110',
  'D107',
  'L110',
  'S112'],
 'paratope_residues': ['E46',
  'A47',
  'A62',
  'D63',
  'S64',
  'P104',
  'S106',
  'D107',
  'L109',
  'L110',
  'S112',
  'S113'],
 'paratope_distances': [4.932139439559644,
  4.882703768329026,
  3.259092485663634,
  3.4500760448654426,
  4.937698001099049,
  3.7328182762800606,
  2.7155682859716204,
  2.857378842525704,
  3.815882991351919,
  3.111731000790443,
  3.7035134681724515,
  4.213256212616139],
 'paratope_closest_partner': ['E136',
  'E136',
  'E136',
  'E13

# protenix